In [1]:
import torch
import torch.nn as nn

x = torch.tensor([2.0, 6.0])

ortalama = x.mean()
std = x.std(unbiased=False)
print(ortalama, std)

print((x - ortalama) / std)

tensor(4.) tensor(2.)
tensor([-1.,  1.])


In [2]:
ln = nn.LayerNorm(2)

print(ln(x))
print(ln.weight)
print(ln.bias)

tensor([-1.0000,  1.0000], grad_fn=<NativeLayerNormBackward0>)
Parameter containing:
tensor([1., 1.], requires_grad=True)
Parameter containing:
tensor([0., 0.], requires_grad=True)


In [3]:
ln_buyuk = nn.LayerNorm(256)
print(ln_buyuk.weight.shape)
print(ln_buyuk.bias.shape)

torch.Size([256])
torch.Size([256])


In [4]:
batch = torch.randn(2, 5, 8) * 100
ln8 = nn.LayerNorm(8)

out = ln8(batch)
print(batch.shape, out.shape)

print(batch[0, 0])
print(out[0, 0])

torch.Size([2, 5, 8]) torch.Size([2, 5, 8])
tensor([  11.4266,  127.9168, -230.2056,   74.2502,  -28.3477,  -55.0987,
        -132.2604, -102.8544])
tensor([ 0.4949,  1.5761, -1.7477,  1.0780,  0.1258, -0.1225, -0.8387, -0.5658],
       grad_fn=<SelectBackward0>)


## MLP


In [5]:
import torch.nn.functional as F

x = torch.tensor([-3.0, 2.0, -1.0, 5.0])
print(torch.relu(x))
print(F.gelu(x))

tensor([0., 2., 0., 5.])
tensor([-4.0499e-03,  1.9545e+00, -1.5866e-01,  5.0000e+00])


In [6]:
torch.manual_seed(0)
token = torch.randn(8)

genislet = nn.Linear(8, 32)
daralt = nn.Linear(32, 8)

h = genislet(token)
print("genişletince:", h.shape)

h = F.gelu(h)
print("GELU sonrası:", h.shape)

cikti = daralt(h)
print("daraltınca:  ", cikti.shape)

genişletince: torch.Size([32])
GELU sonrası: torch.Size([32])
daraltınca:   torch.Size([8])


In [7]:
batch = torch.randn(2, 5, 8)

h = genislet(batch)
print("genişletince:", h.shape)

h = F.gelu(h)
cikti = daralt(h)
print("daraltınca:  ", cikti.shape)

genişletince: torch.Size([2, 5, 32])
daraltınca:   torch.Size([2, 5, 8])


In [8]:
class MLP(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()
        self.genislet = nn.Linear(embedding_dim, 4 * embedding_dim)
        self.daralt = nn.Linear(4 * embedding_dim, embedding_dim)

    def forward(self, x):
        h = self.genislet(x)
        h = F.gelu(h)
        return self.daralt(h)

mlp = MLP(8)
print(mlp(batch).shape)

torch.Size([2, 5, 8])


### attention  - decoder block


In [11]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embedding_dim, head_dim):
        super().__init__()
        self.W_q = nn.Linear(embedding_dim, head_dim, bias=False)
        self.W_k = nn.Linear(embedding_dim, head_dim, bias=False)
        self.W_v = nn.Linear(embedding_dim, head_dim, bias=False)

    def forward(self, x):
        seq_len = x.shape[-2]

        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        scores = q @ k.transpose(-2, -1)

        mask = torch.tril(torch.ones(seq_len, seq_len, device=x.device))
        scores = scores.masked_fill(mask == 0, float("-inf"))

        weights = F.softmax(scores / k.shape[-1] ** 0.5, dim=-1)
        return weights @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        head_dim = embedding_dim // num_heads
        self.heads = nn.ModuleList(
            [CausalSelfAttention(embedding_dim, head_dim) for _ in range(num_heads)]
        )
        self.W_o = nn.Linear(embedding_dim, embedding_dim, bias=False)

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.W_o(out)


torch.manual_seed(0)
mha = MultiHeadAttention(embedding_dim=8, num_heads=4)

batch = torch.randn(2, 5, 8)

print(mha.heads[0](batch).shape)
print(mha(batch).shape)

torch.Size([2, 5, 2])
torch.Size([2, 5, 8])


In [12]:
torch.manual_seed(0)
x = torch.randn(2, 5, 8)

ln1 = nn.LayerNorm(8)
attn = MultiHeadAttention(8, 4)
ln2 = nn.LayerNorm(8)
mlp = MLP(8)

x = x + attn(ln1(x))
print("attention sonrası:", x.shape)

x = x + mlp(ln2(x))
print("MLP sonrası:      ", x.shape)

attention sonrası: torch.Size([2, 5, 8])
MLP sonrası:       torch.Size([2, 5, 8])


In [13]:
class DecoderBlock(nn.Module):
    def __init__(self, embedding_dim, num_heads):
        super().__init__()
        self.ln1 = nn.LayerNorm(embedding_dim)
        self.attn = MultiHeadAttention(embedding_dim, num_heads)
        self.ln2 = nn.LayerNorm(embedding_dim)
        self.mlp = MLP(embedding_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

blok = DecoderBlock(8, 4)
print(blok(batch).shape)

torch.Size([2, 5, 8])
